In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/puttadatta/fintech-project/unified_dev_sample.parquet
/kaggle/input/notebooks/puttadatta/fintech-project/__results__.html
/kaggle/input/notebooks/puttadatta/fintech-project/unified_user_exposure.parquet
/kaggle/input/notebooks/puttadatta/fintech-project/__notebook__.ipynb
/kaggle/input/notebooks/puttadatta/fintech-project/__output__.json
/kaggle/input/notebooks/puttadatta/fintech-project/custom.css


In [2]:
import pandas as pd

In [3]:
dev_df = pd.read_parquet("/kaggle/input/notebooks/puttadatta/fintech-project/unified_dev_sample.parquet")
dev_df.head()

,user_id,step,amount,balance_after,isFraud,role,rolling_peak_3
0,C1000001725,46,-8424.740234,0.0,0,origin,0.0
1,C1000018372,156,-108365.468750,18449576.0,0,origin,0.0
2,C1000022742,18,-2368.679932,0.0,0,origin,0.0
3,C1000069522,329,-590353.812500,13756703.0,0,origin,0.0
4,C1000089317,140,-283090.375000,0.0,0,origin,0.0


In [4]:
dev_df.shape

(418269, 7)

In [5]:
dev_df.describe()

,step,amount,balance_after,isFraud,rolling_peak_3
count,418269.000000,4.182690e+05,4.182690e+05,418269.000000,418269.000000
mean,243.768118,-1.056744e+03,1.036228e+06,0.001308,0.028824
std,142.285046,5.957035e+05,3.114764e+06,0.036140,0.129388
min,1.000000,-5.894475e+07,0.000000e+00,0.000000,0.000000
25%,156.000000,-7.604440e+04,0.000000e+00,0.000000,0.000000
50%,240.000000,-3.064300e+02,1.166264e+04,0.000000,0.000000
75%,335.000000,7.228514e+04,5.507579e+05,0.000000,0.000000
max,743.000000,5.957950e+07,2.059364e+08,1.000000,1.000000


In [6]:
dev_df['rolling_peak_3'].quantile([0.90,.95,.99])

0.90    0.000000
0.95    0.165545
0.99    0.828843
Name: rolling_peak_3, dtype: float64

In [7]:
dev_df = dev_df.rename(columns={'rolling_peak_3':'drawdown_ratio'})
dev_df.columns

Index(['user_id', 'step', 'amount', 'balance_after', 'isFraud', 'role',
       'drawdown_ratio'],
      dtype='object')

In [8]:
per_drawdown = 100*(dev_df['drawdown_ratio']>.2).sum()/len(dev_df)
print("Transactions that have more than 20% drawdown are:",per_drawdown)

Transactions that have more than 20% drawdown are: 4.525795600438952


In [9]:
high_dd = dev_df['drawdown_ratio']>.2
fraud_rate_in20 = dev_df.loc[high_dd,'isFraud'].mean()
low_dd = dev_df['drawdown_ratio']<=.2
fraud_rate_out20 = dev_df.loc[low_dd,'isFraud'].mean()
print("fraud rate below 20% of drawdown ratio:",fraud_rate_in20)
print("fraud rate above 20% of drawdown ratio:",fraud_rate_out20)

fraud rate below 20% of drawdown ratio: 0.0
fraud rate above 20% of drawdown ratio: 0.001369763534240332


In [10]:
dev_df['high_drawdown_flag'] = (dev_df['drawdown_ratio']>.20)

In [11]:
step_risk_table = (
    dev_df
    .groupby('step')
    .agg(
        transaction_count = ('user_id','count'),
        drawdown_rate = ('high_drawdown_flag','mean'),
        fraud_rate = ('isFraud','mean'),
        avg_drawdown = ('drawdown_ratio','mean')
    )
)

step_risk_table.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown
step,,,,
1,180,0.011111,0.0,0.010098
2,61,0.049180,0.0,0.029789
3,31,0.064516,0.0,0.030009
4,28,0.071429,0.0,0.037322
5,46,0.130435,0.0,0.063839


In [12]:
step_risk_table_2 = step_risk_table.copy()
step_risk_table_3 = step_risk_table.copy()

In [13]:
step_risk_table_2 = step_risk_table_2.sort_values(
    by = ['drawdown_rate'],
    ascending = False
)
step_risk_table_2.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown
step,,,,
445,4,0.250000,0.000000,0.242620
483,16,0.187500,0.000000,0.127306
630,11,0.181818,0.000000,0.087587
338,23,0.173913,0.000000,0.112854
531,19,0.157895,0.052632,0.135037


In [14]:
step_risk_table_3 = step_risk_table_3.sort_values(
    by = ['fraud_rate'],
    ascending = False
)
step_risk_table_3.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown
step,,,,
743,1,0.0,1.0,0.0
742,1,0.0,1.0,0.0
713,1,0.0,1.0,0.0
711,2,0.0,1.0,0.0
703,1,0.0,1.0,0.0


In [15]:
step_risk_table['transaction_count'].quantile([0.40,0.50,.75,.90,.95,.99])

0.40      52.00
0.50     147.00
0.75    1607.00
0.90    2201.00
0.95    2503.25
0.99    3084.00
Name: transaction_count, dtype: float64

In [16]:
step_risk_table = step_risk_table[step_risk_table['transaction_count']>=100]
step_risk_table.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown
step,,,,
1,180,0.011111,0.000000,0.010098
6,118,0.033898,0.008475,0.016880
7,493,0.008114,0.002028,0.006816
8,1368,0.004386,0.000731,0.003510
9,2398,0.018349,0.000834,0.013338


In [17]:
overall_drawdown_rate_avg = step_risk_table['drawdown_rate'].mean()
overall_fraud_rate_avg = step_risk_table['fraud_rate'].mean()
print("the overall average drawdown rate is:",overall_drawdown_rate_avg)
print("the overall average fraud rate is:",overall_fraud_rate_avg)

the overall average drawdown rate is: 0.04496847132064381
the overall average fraud rate is: 0.0013107066706368861


In [18]:
step_risk_table['drawdown_amp'] = (step_risk_table['drawdown_rate']/overall_drawdown_rate_avg)
step_risk_table['fraud_amp'] = (step_risk_table['fraud_rate']/overall_fraud_rate_avg)

In [19]:
step_risk_table['riskindx'] = step_risk_table['drawdown_amp'] + step_risk_table['fraud_amp']

In [20]:
step_risk_table = step_risk_table.sort_values(
    by = ['riskindx'],
    ascending = False
)
step_risk_table.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown,drawdown_amp,fraud_amp,riskindx
step,,,,,,,
25,105,0.057143,0.019048,0.034017,1.270732,14.532328,15.803059
617,172,0.063953,0.017442,0.040430,1.422185,13.307219,14.729404
428,176,0.045455,0.017045,0.035359,1.010809,13.004782,14.015591
670,137,0.043796,0.014599,0.027283,0.973918,11.137916,12.111834
659,174,0.080460,0.011494,0.050684,1.789249,8.769508,10.558757


In [21]:
step_risk_table.shape

(317, 7)

In [22]:
highrisk_df = step_risk_table[['transaction_count','drawdown_rate','fraud_rate','drawdown_amp','fraud_amp','riskindx']]
highrisk_df.head()

,transaction_count,drawdown_rate,fraud_rate,drawdown_amp,fraud_amp,riskindx
step,,,,,,
25,105,0.057143,0.019048,1.270732,14.532328,15.803059
617,172,0.063953,0.017442,1.422185,13.307219,14.729404
428,176,0.045455,0.017045,1.010809,13.004782,14.015591
670,137,0.043796,0.014599,0.973918,11.137916,12.111834
659,174,0.080460,0.011494,1.789249,8.769508,10.558757


In [23]:
highrisk_df.shape

(317, 6)

In [24]:
highrisk_df.to_parquet('highrisk_df.parquet', index = False)